# Evaluation

In [1]:
import os, glob, shutil

copied = 0
for src in glob.glob("/kaggle/input/**/*", recursive=True):
    if os.path.isdir(src):
        continue
    if src.endswith((".py", ".json", ".pt")):
        rel = src
        name = "eval/metrics.py" if src.endswith("eval/metrics.py") else os.path.basename(src)
        os.makedirs(os.path.dirname(name), exist_ok=True) if os.path.dirname(name) else None
        shutil.copy(src, name)
        copied += 1
print(f"copied {copied} files from /kaggle/input")
print("working dir now has:", sorted(f for f in os.listdir(".") if f.endswith((".py",".json",".pt"))))

copied 7 files from /kaggle/input
working dir now has: ['best_model.pt', 'passages.json', 'queries_meta.json', 'test_eval.json', 'tokenizer.json', 'train_pairs.json', 'val_eval.json']


In [2]:
!pip install -q rank_bm25 scikit-learn

In [3]:
import os
os.makedirs("eval", exist_ok=True)
print("created eval/ package dir")


created eval/ package dir


In [4]:
%%writefile config.py
import os
import torch


def _int(name, default):
    return int(os.environ.get(name, default))


def _flt(name, default):
    return float(os.environ.get(name, default))


SEED = _int("NS_SEED", 42)

DATA_DIR          = os.environ.get("NS_DATA_DIR", "data")
TRAIN_PAIRS_PATH  = os.path.join(DATA_DIR, "train_pairs.json")
VAL_EVAL_PATH     = os.path.join(DATA_DIR, "val_eval.json")
TEST_EVAL_PATH    = os.path.join(DATA_DIR, "test_eval.json")
PASSAGES_PATH     = os.path.join(DATA_DIR, "passages.json")
QUERIES_META_PATH = os.path.join(DATA_DIR, "queries_meta.json")
TOKENIZER_PATH    = os.environ.get("NS_TOKENIZER", "tokenizer.json")
MODEL_PATH        = os.environ.get("NS_MODEL", "best_model.pt")

MAX_EXAMPLES = _int("NS_MAX_EXAMPLES", 50_000)
VOCAB_SIZE   = _int("NS_VOCAB_SIZE", 8000)

D_MODEL  = _int("NS_D_MODEL", 256)
NHEAD    = _int("NS_NHEAD", 4)
NUM_LAYERS = _int("NS_NUM_LAYERS", 4)
DIM_FF   = _int("NS_DIM_FF", 512)
DROPOUT  = _flt("NS_DROPOUT", 0.1)
MAX_LEN  = _int("NS_MAX_LEN", 128)
MODEL_MAX_LEN = _int("NS_MODEL_MAX_LEN", 512)

# Run 2: batch 64->128 (more in-batch negatives), lr 1e-4->2e-4 (scaled for batch),
# epochs 10->15 (keep gradient-update count comparable to Run 1's 3,820).
FINETUNE_EPOCHS = _int("NS_FINETUNE_EPOCHS", 15)
FINETUNE_BS     = _int("NS_FINETUNE_BS", 128)
FINETUNE_LR     = _flt("NS_FINETUNE_LR", 2e-4)
TEMPERATURE     = _flt("NS_TEMPERATURE", 0.05)

EVAL_K     = _int("NS_EVAL_K", 10)
ENCODE_BS  = _int("NS_ENCODE_BS", 512)

DEVICE = os.environ.get("NS_DEVICE", "cuda" if torch.cuda.is_available() else "cpu")

Writing config.py


In [5]:
%%writefile model.py
import math
import torch
import torch.nn as nn
import torch.nn.functional as F


def masked_mean(hidden, attention_mask):
    mask = attention_mask.unsqueeze(-1).float()
    summed = (hidden * mask).sum(dim=1)
    counts = mask.sum(dim=1).clamp(min=1e-9)
    return summed / counts


class PositionalEncoding(nn.Module):
    def __init__(self, d_model, max_len=512):
        super().__init__()
        pe = torch.zeros(max_len, d_model)
        pos = torch.arange(0, max_len, dtype=torch.float).unsqueeze(1)
        div = torch.exp(torch.arange(0, d_model, 2).float() * (-math.log(10000.0) / d_model))
        pe[:, 0::2] = torch.sin(pos * div)
        pe[:, 1::2] = torch.cos(pos * div)
        self.register_buffer("pe", pe.unsqueeze(0))

    def forward(self, x):
        return x + self.pe[:, : x.size(1)]


class MultiHeadSelfAttention(nn.Module):
    def __init__(self, d_model, nhead, dropout=0.1):
        super().__init__()
        assert d_model % nhead == 0
        self.nhead = nhead
        self.d_head = d_model // nhead
        self.q_proj = nn.Linear(d_model, d_model)
        self.k_proj = nn.Linear(d_model, d_model)
        self.v_proj = nn.Linear(d_model, d_model)
        self.out_proj = nn.Linear(d_model, d_model)
        self.dropout = nn.Dropout(dropout)

    def forward(self, x, key_padding_mask):
        B, L, d = x.shape
        def split(t):
            return t.view(B, L, self.nhead, self.d_head).transpose(1, 2)
        q, k, v = split(self.q_proj(x)), split(self.k_proj(x)), split(self.v_proj(x))
        scores = (q @ k.transpose(-2, -1)) / math.sqrt(self.d_head)
        if key_padding_mask is not None:
            scores = scores.masked_fill(key_padding_mask.view(B, 1, 1, L), float("-inf"))
        attention = self.dropout(torch.softmax(scores, dim=-1))
        ctx = (attention @ v).transpose(1, 2).contiguous().view(B, L, d)
        return self.out_proj(ctx)


class TransformerBlock(nn.Module):
    def __init__(self, d_model, nhead, dim_feedforward, dropout=0.1):
        super().__init__()
        self.norm1 = nn.LayerNorm(d_model)
        self.attn = MultiHeadSelfAttention(d_model, nhead, dropout)
        self.norm2 = nn.LayerNorm(d_model)
        self.ff = nn.Sequential(
            nn.Linear(d_model, dim_feedforward),
            nn.GELU(),
            nn.Dropout(dropout),
            nn.Linear(dim_feedforward, d_model),
        )
        self.dropout = nn.Dropout(dropout)

    def forward(self, x, key_padding_mask):
        x = x + self.dropout(self.attn(self.norm1(x), key_padding_mask))
        x = x + self.dropout(self.ff(self.norm2(x)))
        return x


class TransformerEncoderModel(nn.Module):
    def __init__(self, vocab_size, d_model=256, nhead=4, num_layers=4,
                 dim_feedforward=512, max_len=512, dropout=0.1, pad_id=0):
        super().__init__()
        self.pad_id = pad_id
        self.d_model = d_model
        self.embed = nn.Embedding(vocab_size, d_model, padding_idx=pad_id)
        self.pos = PositionalEncoding(d_model, max_len)
        self.in_dropout = nn.Dropout(dropout)
        self.blocks = nn.ModuleList([
            TransformerBlock(d_model, nhead, dim_feedforward, dropout)
            for _ in range(num_layers)
        ])
        self.final_norm = nn.LayerNorm(d_model)

    def forward(self, input_ids, attention_mask):
        x = self.embed(input_ids) * math.sqrt(self.d_model)
        x = self.in_dropout(self.pos(x))
        key_padding_mask = attention_mask == 0
        for block in self.blocks:
            x = block(x, key_padding_mask)
        x = self.final_norm(x)
        return masked_mean(x, attention_mask)


class BagOfEmbeddings(nn.Module):
    def __init__(self, vocab_size, d_model=256, pad_id=0, dropout=0.1):
        super().__init__()
        self.pad_id = pad_id
        self.d_model = d_model
        self.embed = nn.Embedding(vocab_size, d_model, padding_idx=pad_id)
        self.dropout = nn.Dropout(dropout)

    def forward(self, input_ids, attention_mask):
        return masked_mean(self.dropout(self.embed(input_ids)), attention_mask)

Writing model.py


In [6]:
%%writefile data.py
from typing import Callable, List, Tuple
import torch
from torch import Tensor
from torch.utils.data import Dataset
import config


class TripletDataset(Dataset):
    def __init__(self, triples, tokenize: Callable, max_length: int = config.MAX_LEN):
        self.triples = triples
        self.tokenize = tokenize
        self.max_length = max_length

    def __len__(self) -> int:
        return len(self.triples)

    def __getitem__(self, i):
        q, pos, neg = self.triples[i]
        m = self.max_length
        return self.tokenize(q)[:m], self.tokenize(pos)[:m], self.tokenize(neg)[:m]


class TextDataset(Dataset):
    def __init__(self, texts, tokenize: Callable, max_length: int = config.MAX_LEN):
        self.texts = texts
        self.tokenize = tokenize
        self.max_length = max_length

    def __len__(self) -> int:
        return len(self.texts)

    def __getitem__(self, i) -> List[int]:
        return self.tokenize(self.texts[i])[:self.max_length]


def pad_batch(sequences: List[List[int]], pad_id: int = 0) -> Tuple[Tensor, Tensor]:
    sequences = [seq if seq else [pad_id] for seq in sequences]
    max_len = max(len(s) for s in sequences)
    ids  = torch.full((len(sequences), max_len), pad_id, dtype=torch.long)
    mask = torch.zeros((len(sequences), max_len), dtype=torch.long)
    for i, seq in enumerate(sequences):
        ids[i, :len(seq)]  = torch.tensor(seq, dtype=torch.long)
        mask[i, :len(seq)] = 1
    return ids, mask


def triplet_collate(batch, pad_id: int = 0):
    queries, pos, neg = zip(*batch)
    return pad_batch(queries, pad_id), pad_batch(pos, pad_id), pad_batch(neg, pad_id)


def text_collate(batch, pad_id: int = 0) -> Tuple[Tensor, Tensor]:
    return pad_batch(batch, pad_id)

Writing data.py


In [7]:
%%writefile build_tokenizer.py
import json, os, tempfile
from typing import Callable, List, Tuple
from tokenizers import Tokenizer, models, trainers, pre_tokenizers, decoders
import config

SPECIAL_TOKENS = ["[PAD]", "[UNK]", "[CLS]", "[SEP]", "[MASK]"]


def _text_iterator():
    with open(config.QUERIES_META_PATH, encoding="utf-8") as f:
        for item in json.load(f):
            yield item["query"]
    with open(config.PASSAGES_PATH, encoding="utf-8") as f:
        for item in json.load(f):
            yield item["text"]


def train_tokenizer(vocab_size: int = config.VOCAB_SIZE,
                    out_path: str = config.TOKENIZER_PATH) -> Tokenizer:
    tokenizer = Tokenizer(models.BPE(unk_token="[UNK]"))
    tokenizer.pre_tokenizer = pre_tokenizers.Whitespace()
    tokenizer.decoder = decoders.BPEDecoder()
    trainer = trainers.BpeTrainer(vocab_size=vocab_size, special_tokens=SPECIAL_TOKENS)

    with tempfile.NamedTemporaryFile(mode="w", suffix=".txt",
                                     delete=False, encoding="utf-8") as tmp:
        for text in _text_iterator():
            tmp.write(text.replace("\n", " ") + "\n")
        tmp_path = tmp.name

    tokenizer.train([tmp_path], trainer)
    os.unlink(tmp_path)
    tokenizer.save(out_path)
    print(f"saved tokenizer -> {out_path} (vocab={tokenizer.get_vocab_size()})")
    return tokenizer


def load_tokenizer(path: str = config.TOKENIZER_PATH) -> Tuple[Callable[[str], List[int]], int, int]:
    tokenizer = Tokenizer.from_file(path)
    pad_id = tokenizer.token_to_id("[PAD]")
    vocab_size = tokenizer.get_vocab_size()
    tokenize = lambda text: tokenizer.encode(text).ids
    return tokenize, pad_id, vocab_size


def main() -> None:
    train_tokenizer()

Writing build_tokenizer.py


In [8]:
%%writefile eval/metrics.py
from typing import List


def recall_at_k(retrieved_ids: List[int], relevant_id: int, k: int) -> float:
    return 1.0 if relevant_id in retrieved_ids[:k] else 0.0


def reciprocal_rank(retrieved_ids: List[int], relevant_id: int) -> float:
    for i, rid in enumerate(retrieved_ids):
        if rid == relevant_id:
            return 1.0 / (i + 1)
    return 0.0


def mean_reciprocal_rank(retrieved_list: List[List[int]], gold_ids: List[int]) -> float:
    scores = [reciprocal_rank(ret, gold) for ret, gold in zip(retrieved_list, gold_ids)]
    return sum(scores) / len(scores) if scores else 0.0


def macro_recall_at_k(retrieved_list: List[List[int]], gold_ids: List[int], k: int) -> float:
    scores = [recall_at_k(ret, gold, k) for ret, gold in zip(retrieved_list, gold_ids)]
    return sum(scores) / len(scores) if scores else 0.0


Writing eval/metrics.py


In [9]:
%%writefile eval/baselines.py


import re
from typing import List

from rank_bm25 import BM25Okapi 
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.metrics.pairwise import cosine_similarity


def _simple_tokenize(text: str) -> List[str]:
    return re.findall(r"[a-z0-9]+", text.lower())


class BM25Search:

    def __init__(self, passages):
        self.passages = passages
        corpus = [_simple_tokenize(p["text"]) for p in passages]
        self.bm25 = BM25Okapi(corpus)

    def retrieve(self, query: str, k: int) -> List[int]:
        scores = self.bm25.get_scores(_simple_tokenize(query))
        top = scores.argsort()[::-1][:k]
        return [self.passages[i]["id"] for i in top]


class TfidfSearch:

    def __init__(self, passages):
        self.passages = passages
        self.vectorizer = TfidfVectorizer(
            lowercase=True,
            stop_words="english",
            ngram_range=(1, 2),
            sublinear_tf=True,
        )
        self.doc_matrix = self.vectorizer.fit_transform(p["text"] for p in passages)

    def retrieve(self, query: str, k: int) -> List[int]:
        q_vec = self.vectorizer.transform([query])
        scores = cosine_similarity(q_vec, self.doc_matrix)[0]
        top = scores.argsort()[::-1][:k]
        return [self.passages[i]["id"] for i in top]

Writing eval/baselines.py


In [10]:
%%writefile evaluate.py

import json

import torch
import torch.nn.functional as F
from torch.utils.data import DataLoader

import config

try:
    from data.build_tokenizer import load_tokenizer
except ImportError: 
    from build_tokenizer import load_tokenizer

try:
    from model.model import TransformerEncoderModel
except ImportError:
    from model import TransformerEncoderModel

try:
    from eval.metrics import macro_recall_at_k, mean_reciprocal_rank
except ImportError:
    from metrics import macro_recall_at_k, mean_reciprocal_rank

try:
    from eval.baselines import BM25Search, TfidfSearch
except ImportError:
    from baselines import BM25Search, TfidfSearch

from data import TextDataset, text_collate


def load_eval_set(path):
    with open(path, encoding="utf-8") as f:
        data = json.load(f)
    return data["queries"], data["gold"]


def _default_passages_path():
    for attr in ("PASSAGES_PATH", "CHUNKS_PATH"):
        p = getattr(config, attr, None)
        if p:
            return p
    return "passages.json"


def load_passages(path=None):
    path = path or _default_passages_path()
    with open(path, encoding="utf-8") as f:
        return json.load(f)


class NeuralRetriever:


    def __init__(self, passages):
        self.passages = passages
        self.tokenize, self.pad_id, vocab_size = load_tokenizer()
        self.device = torch.device(config.DEVICE)
        self.model = TransformerEncoderModel(
            vocab_size=vocab_size,
            d_model=config.D_MODEL,
            nhead=config.NHEAD,
            num_layers=config.NUM_LAYERS,
            dim_feedforward=config.DIM_FF,
            max_len=config.MODEL_MAX_LEN,
            dropout=config.DROPOUT,
            pad_id=self.pad_id,
        ).to(self.device)
        self.model.load_state_dict(torch.load(config.MODEL_PATH, map_location=self.device))
        self.model.eval()
        self.index = None 

    def build_index(self, batch_size=None):
        batch_size = batch_size or getattr(config, "ENCODE_BS", 256)
        texts = [p["text"] for p in self.passages]
        loader = DataLoader(
            TextDataset(texts, self.tokenize),
            batch_size=batch_size,
            collate_fn=lambda b: text_collate(b, self.pad_id),
        )
        embeddings = []
        with torch.no_grad():
            for input_ids, attention_mask in loader:
                emb = self.model(input_ids.to(self.device), attention_mask.to(self.device))
                embeddings.append(emb.cpu())
        self.index = F.normalize(torch.cat(embeddings, dim=0), dim=-1)
        print(f"neural index: {self.index.shape[0]:,} passages x {self.index.shape[1]} dims")
        return self

    def retrieve(self, query: str, k: int) -> list:
        assert self.index is not None, "call build_index() first"
        with torch.no_grad():
            ids = torch.tensor(
                [self.tokenize(query)[: config.MAX_LEN]], dtype=torch.long
            ).to(self.device)
            mask = torch.ones_like(ids)
            q_emb = F.normalize(self.model(ids, mask), dim=-1).cpu()
        top = (q_emb @ self.index.T)[0].topk(k).indices.tolist()
        return [self.passages[i]["id"] for i in top]


def evaluate_retriever(retriever, queries, gold, ks=(1, 5, 10)):
    max_k = max(ks)
    retrieved = [retriever.retrieve(q, max_k) for q in queries]
    results = {f"Recall@{k}": macro_recall_at_k(retrieved, gold, k) for k in ks}
    results["MRR"] = mean_reciprocal_rank(retrieved, gold)
    return results


def print_table(scores_by_name, ks=(1, 5, 10)):
    cols = [f"Recall@{k}" for k in ks] + ["MRR"]
    header = f"{'Retriever':<16}" + "".join(f"{c:>11}" for c in cols)
    print(header)
    print("-" * len(header))
    for name, scores in scores_by_name.items():
        row = f"{name:<16}" + "".join(f"{scores[c]:>11.4f}" for c in cols)
        print(row)


def main(eval_path=config.TEST_EVAL_PATH, save_path="eval_results.json", max_queries=None):
    passages = load_passages()
    queries, gold = load_eval_set(eval_path)
    if max_queries: 
        queries, gold = queries[:max_queries], gold[:max_queries]
    print(f"corpus: {len(passages):,} passages | eval queries: {len(queries):,}\n")

    retrievers = {
        "BM25": BM25Search(passages),
        "TF-IDF": TfidfSearch(passages),
        "Neural (ours)": NeuralRetriever(passages).build_index(),
    }

    scores_by_name = {
        name: evaluate_retriever(r, queries, gold) for name, r in retrievers.items()
    }

    print()
    print_table(scores_by_name)

    if save_path:
        with open(save_path, "w", encoding="utf-8") as f:
            json.dump(scores_by_name, f, indent=2)
        print(f"\nsaved -> {save_path}")

    return scores_by_name


if __name__ == "__main__":
    main()

Writing evaluate.py


In [11]:
%%writefile demo.py

import sys

import config

try:
    from eval.evaluate import NeuralRetriever, load_passages
except ImportError:
    from evaluate import NeuralRetriever, load_passages


def make_search():
    passages = load_passages()
    text_lookup = {p["id"]: p["text"] for p in passages}
    retriever = NeuralRetriever(passages).build_index()

    def search(query: str, k: int = 5, preview: int = 280):
        print(f"\nQuery: {query}\n" + "=" * 70)
        for rank, pid in enumerate(retriever.retrieve(query, k), start=1):
            snippet = text_lookup[pid][:preview].replace("\n", " ")
            print(f"[{rank}] passage {pid}: {snippet}...\n")

    return search


if __name__ == "__main__":
    query = " ".join(sys.argv[1:]) or "what is a transformer and how does attention work"
    search = make_search()
    search(query)

Writing demo.py


In [12]:
import importlib, os
need_files = ["passages.json", "test_eval.json", "tokenizer.json", "best_model.pt"]
need_mods  = ["config", "build_tokenizer", "data", "model"]
missing = [f for f in need_files if not os.path.exists(f) and not os.path.exists(os.path.join("data", f))]
for m in need_mods:
    try:
        importlib.import_module(m)
    except Exception as e:
        missing.append(f"module {m} ({e})")
print("MISSING:", missing if missing else "nothing — good to go")

MISSING: nothing — good to go


In [14]:
import os, shutil
os.makedirs("data", exist_ok=True)
for f in ["passages.json", "queries_meta.json", "test_eval.json", "train_pairs.json", "val_eval.json"]:
    if os.path.exists(f):
        shutil.copy(f, "data/" + f)
print("data/ now has:", os.listdir("data"))

data/ now has: ['train_pairs.json', 'test_eval.json', 'val_eval.json', 'queries_meta.json', 'passages.json']


In [17]:
import json, random
random.seed(42)

with open("data/passages.json") as f:
    passages = json.load(f)
with open("data/test_eval.json") as f:
    ev = json.load(f)

gold_ids = set(ev["gold"])
CAP = 100_000
others = [p for p in passages if p["id"] not in gold_ids]
random.shuffle(others)
keep = [p for p in passages if p["id"] in gold_ids] + others[:CAP]
random.shuffle(keep)

with open("data/passages.json", "w") as f:
    json.dump(keep, f)
print(f"reduced corpus: {len(keep):,} passages (kept all {len(gold_ids):,} gold + {CAP:,} distractors)")

reduced corpus: 112,229 passages (kept all 12,229 gold + 100,000 distractors)


In [18]:
import evaluate
scores = evaluate.main(max_queries=500)  

corpus: 112,229 passages | eval queries: 500

neural index: 112,229 passages x 256 dims

Retriever          Recall@1   Recall@5  Recall@10        MRR
------------------------------------------------------------
BM25                 0.4540     0.6900     0.7680     0.5575
TF-IDF               0.4060     0.6620     0.7420     0.5154
Neural (ours)        0.2640     0.4420     0.5040     0.3384

saved -> eval_results.json


In [21]:
import os, re, json, urllib.request

os.makedirs("book", exist_ok=True)
chapters = {
    "2": "https://web.stanford.edu/~jurafsky/slp3/2.pdf",   # tokenization
    "6": "https://web.stanford.edu/~jurafsky/slp3/6.pdf",   # vector semantics & embeddings
    "9": "https://web.stanford.edu/~jurafsky/slp3/9.pdf",   # transformers
}
for name, url in chapters.items():
    p = f"book/{name}.pdf"
    if not os.path.exists(p):
        urllib.request.urlretrieve(url, p); print("downloaded", p)

try:
    import fitz
except ImportError:
    os.system("pip install -q pymupdf"); import fitz
text = "\n".join(" ".join(pg.get_text() for pg in fitz.open(f"book/{n}.pdf")) for n in chapters)

def sents(t):
    t = re.sub(r"\s+", " ", t).strip()
    return [s.strip() for s in re.split(r"(?<=[.!?])\s+", t) if s.strip()]
def chunk(t, lo=200, hi=300):
    out, cur, n = [], [], 0
    for s in sents(t):
        w = len(s.split())
        if n + w > hi and n >= lo:
            out.append(" ".join(cur)); cur, n = [], 0
        cur.append(s); n += w
    if cur and n >= lo // 4: out.append(" ".join(cur))
    return out
book = [{"id": i, "text": c} for i, c in enumerate(chunk(text))]
with open("book_chunks.json", "w") as f: json.dump(book, f)
print(f"book corpus: {len(book)} passages from {len(chapters)} J&M chapters")

from evaluate import NeuralRetriever
retriever = NeuralRetriever(book).build_index()
lookup = {p["id"]: p["text"] for p in book}
def search(query, k=3, preview=280):
    print(f"\nQuery: {query}\n" + "=" * 70)
    for rank, pid in enumerate(retriever.retrieve(query, k), 1):
        print(f"[{rank}] {lookup[pid][:preview]}...\n")

for q in ["what is the transformer attention mechanism",
          "how are word embeddings learned",
          "what is tokenization"]:
    search(q)

downloaded book/2.pdf
downloaded book/6.pdf
downloaded book/9.pdf
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 25.0/25.0 MB 76.4 MB/s eta 0:00:00
book corpus: 132 passages from 3 J&M chapters
neural index: 132 passages x 256 dims

Query: what is the transformer attention mechanism
[1] Imagine some string (perhaps it is exention) that is in this optimal path (whatever it is). The intuition of dynamic programming is that if exention is in the optimal operation list, then the optimal sequence must also include the optimal path from intention to exention. Why? If ...

[2] The other is content words: nouns, adjectives and verbs that tend content words to have meanings about people and places and events. Nouns, and especially partic- ular nouns like names and technical terms do tend to grow indeﬁnitely. So models that are sensitive to this differenc...

[3] The Unicode standard is a method for representing text written using any character Unicode in any script of the languages of the world (i

In [22]:
import zipfile
with zipfile.ZipFile("eval_output.zip", "w") as z:
    if os.path.exists("eval_results.json"):
        z.write("eval_results.json")
print("wrote eval_output.zip")

wrote eval_output.zip
